# 임베딩 모델 A/B 테스트

**목적:** 우리 데이터(한국어 복지 정책 + 어르신 말투 질문)에 가장 잘 맞는 임베딩 모델을 고른다.

**진행 순서 (팀 계획)**
1. 현재 청크(v3: 정책 650자 · PDF 700자 · overlap 90)로 **임베딩 모델 A/B 테스트** ← 이 노트북
2. 고른 모델로 **청크 크기 A/B 테스트** (450 / 650 / 900자, overlap 0 / 90 / 150) → `CHUNK_SETS`에 버전 추가해서 같은 노트북으로 진행
3. 청크 데이터 확정 → 최종 임베딩 · Qdrant 적재

> 모델과 청크 크기를 한꺼번에 바꾸면 무엇 때문에 좋아졌는지 알 수 없어서 **한 번에 하나씩** 바꾼다.

**공정한 비교:** 같은 청크, 같은 질문, 같은 k, 같은 지역 필터 → 다른 건 **임베딩 모델 하나뿐**

**평가 지표**

| 지표 | 뜻 |
|---|---|
| Top-1 | 1등 결과가 정답인 비율 |
| Top-3 | 3등 안에 정답이 있는 비율 |
| Top-5 | 5등 안에 정답이 있는 비율 (= 팀 문서의 Recall@5) |
| MRR | 정답 순위 점수 평균 (1등=1, 2등=0.5, 3등=0.33 …, 없음=0) → 높을수록 정답이 위에 나옴 |

> 이 테스트는 **청크 번호(chunk_id)나 UUID가 필요 없다.** 정책 번호와 PDF 문서·쪽으로 채점하기 때문에 청크가 확정되기 전에도 돌릴 수 있다.

## 1. 설정
- `CHUNK_SETS`: 비교할 청크 버전. 지금은 하나, 청크 크기 실험 때 버전을 추가
- `K = 5`: 팀 문서의 Recall@5와 맞춤
- `INCLUDE_TYPES`: 시설 청크는 테스트 질문과 관계없어서 제외 (검색 자리만 차지)
- `SAMPLE_SIZE`: 처음엔 500 정도로 코드가 도는지만 확인, 본 테스트는 `None`

In [ ]:
# 비교할 청크 버전 (지금은 하나, 청크 크기 실험할 때 추가)
CHUNK_SETS = {
    "v3_650_700_ov90": "./data/sample_policy_chunks.jsonl",   # 진짜 청크 오면 경로 교체
    # "v4_450_ov90": "../../data/rag/v4_450/embedding_ready.jsonl",
}
TEST_PATH = "./data/test_questions.jsonl"

K = 5                                   # 몇 등까지 볼지 (Top-5 = Recall@5)
USE_REGION_FILTER = True                # 지역 질문에 지역 필터 (모든 모델 똑같이)
INCLUDE_TYPES = ("policy", "guideline") # 시설 청크는 테스트 질문과 무관해서 제외
SAMPLE_SIZE = None                      # 처음엔 500 정도로 빠르게, 본 테스트는 None

## 2. 청크 읽기
- 강사님 `JSONLoader` 사용
- 합의한 형식은 지역·출처가 `metadata` 안에 한 겹 더 들어 있어서 **펼쳐서** 읽는다
- 청크 버전마다 따로 읽어둔다

In [ ]:
from collections import Counter
from langchain_community.document_loaders import JSONLoader

def metadata_func(record: dict, metadata: dict) -> dict:
    for key, value in record.items():
        if key == "text":
            continue
        if key == "metadata" and isinstance(value, dict):
            metadata.update(value)      # metadata 안의 정보를 펼치기
        else:
            metadata[key] = value
    return metadata

def load_documents(path):
    loader = JSONLoader(
        file_path=path,
        jq_schema=".",
        content_key="text",
        json_lines=True,
        metadata_func=metadata_func,
    )
    docs = loader.load()
    # record_type이 있으면 정책·PDF만 남기기 (없으면 전부 사용)
    docs = [d for d in docs if d.metadata.get("record_type", "policy") in INCLUDE_TYPES]
    if SAMPLE_SIZE:
        docs = docs[:SAMPLE_SIZE]
    return docs

chunk_data = {}
for chunk_name, path in CHUNK_SETS.items():
    chunk_data[chunk_name] = load_documents(path)
    docs = chunk_data[chunk_name]
    print(f"[{chunk_name}] 청크 수: {len(docs)}", Counter(d.metadata.get("record_type") for d in docs))

## 3. 테스트 질문 + 채점할 수 있는 질문만 고르기
- 함정 질문은 정답이 없어서 제외
- **정답이 데이터에 없는 질문도 제외** → 모델 탓이 아닌데 틀린 걸로 세어지는 걸 막기 위해
- ⚠️ 제외 목록을 꼭 확인. 너무 많으면 테스트 질문을 새 데이터에 맞게 바꾼다 (예: 전북 → 전남)

In [ ]:
import json

with open(TEST_PATH, encoding="utf-8") as f:
    all_questions = [json.loads(line) for line in f]

def select_questions(docs):
    # 데이터에 실제로 있는 정답 목록
    policy_ids = {d.metadata.get("policy_id") for d in docs}
    pdf_docs = {d.metadata.get("source") for d in docs if d.metadata.get("record_type") == "guideline"}

    questions, skipped = [], []
    for q in all_questions:
        if q["q_type"] not in ("의미검색", "지역필터", "PDF근거"):
            continue                                   # 함정 질문은 정답이 없어서 제외
        if q["target"] == "policy":
            ok = any(pid in policy_ids for pid in q["expected_ids"])
        else:
            ok = q["expected_document"] in pdf_docs
        (questions if ok else skipped).append(q)
    return questions, skipped

chunk_questions = {}
for chunk_name, docs in chunk_data.items():
    questions, skipped = select_questions(docs)
    chunk_questions[chunk_name] = questions
    print(f"[{chunk_name}] 채점할 질문: {len(questions)}개 / 정답이 데이터에 없어 제외: {len(skipped)}개")
    for q in skipped:
        print("   제외:", q["qid"], q["question"][:25], "| 정답:", q.get("expected_title"))

## 4. 비교할 모델
| 모델 | 고른 이유 |
|---|---|
| qwen3-embedding:0.6b | 수업 모델, 기준 |
| bge-m3 | 무료 다국어 대표 |
| KURE-v1 | bge-m3를 한국어로 추가 학습한 모델, 무료 |
| text-embedding-3-small | 유료 대표, 강사님 `ex02`에서 사용 |

딕셔너리에 한 줄만 추가하면 같은 테스트를 반복할 수 있다.

In [ ]:
from langchain_ollama import OllamaEmbeddings

models = {
    "qwen3-embedding:0.6b": OllamaEmbeddings(model="qwen3-embedding:0.6b"),
    "bge-m3": OllamaEmbeddings(model="bge-m3"),
}

# KURE-v1 (한국어 특화, 무료) / 설치: uv pip install langchain-huggingface sentence-transformers
# from langchain_huggingface import HuggingFaceEmbeddings
# models["KURE-v1"] = HuggingFaceEmbeddings(model_name="nlpai-lab/KURE-v1")

# OpenAI (유료) / .env에 OPENAI_API_KEY 필요
# from dotenv import load_dotenv; load_dotenv()
# from langchain_openai import OpenAIEmbeddings
# models["text-embedding-3-small"] = OpenAIEmbeddings(model="text-embedding-3-small")

## 5. 정답 판정 + 지역 필터
**정답 판정**
- 정책: 정책 하나가 여러 청크(대상 / 혜택 / 신청 …)로 나뉘므로 `chunk_id`가 아니라 **`policy_id`** 로 판정
- PDF: 청크 번호는 청킹할 때마다 바뀌므로 **문서 이름 + 쪽수 범위** 로 판정

**지역 필터**
- "내 시도 정책 + 시도가 비어 있는 정책(전국·미확인)"
- `IsEmpty`를 쓰는 이유: 시도 값이 `null`인 경우뿐 아니라 **칸 자체가 없는 경우**도 잡아야 해서 (`IsNull`은 칸이 없으면 못 잡음)
- 임베딩 비교가 목적이라 필터는 단순하게, 모든 모델에 똑같이

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, IsEmptyCondition, PayloadField

def is_hit(doc, q):
    m = doc.metadata
    # 정책: 정책 번호가 정답 목록에 있으면 정답 (청크 여러 개 중 아무거나)
    if q["target"] == "policy":
        return m.get("policy_id") in q["expected_ids"]
    # PDF: 같은 문서 + 정답 쪽이 청크 쪽 범위 안
    start = m.get("page_start")
    end = m.get("page_end") or start
    if m.get("source") != q["expected_document"] or start is None:
        return False
    return any(start <= p <= end for p in q["expected_pages"])

def region_filter(q):
    if not USE_REGION_FILTER or not q.get("user_sido"):
        return None
    # 내 시도 정책 + 시도가 비어 있는 정책(전국·미확인)
    # IsEmpty = 값이 null이거나, 칸 자체가 없거나, 빈 리스트
    return Filter(should=[
        FieldCondition(key="metadata.sido_name", match=MatchValue(value=q["user_sido"])),
        IsEmptyCondition(is_empty=PayloadField(key="metadata.sido_name")),
    ])

## 6. 모델 하나 평가하기
- 강사님 방식: `QdrantVectorStore.from_documents` + `similarity_search_with_score`
- 테스트용이라 **메모리(`:memory:`)에만** 저장 → Docker Qdrant의 진짜 컬렉션은 건드리지 않음
- 적재 시간, 질문당 검색 시간도 기록

In [ ]:
import time
from langchain_qdrant import QdrantVectorStore

def evaluate(model_name, embedding_model, chunk_name):
    docs = chunk_data[chunk_name]
    questions = chunk_questions[chunk_name]

    # 강사님 코드: from_documents (테스트라 메모리에만 저장)
    t0 = time.time()
    vectorstore = QdrantVectorStore.from_documents(
        documents=docs,
        embedding=embedding_model,
        location=":memory:",
        collection_name="ab_test",
    )
    build_sec = time.time() - t0

    hits = {1: 0, 3: 0, 5: 0}
    rr_sum = 0
    details = []

    t1 = time.time()
    for q in questions:
        results = vectorstore.similarity_search_with_score(
            q["question"], k=K, filter=region_filter(q)
        )

        rank = None
        for i, (doc, score) in enumerate(results, start=1):
            if is_hit(doc, q):
                rank = i
                break

        for n in hits:
            if rank is not None and rank <= n:
                hits[n] += 1
        if rank is not None:
            rr_sum += 1 / rank

        first_doc, first_score = results[0] if results else (None, None)
        details.append({
            "chunk_set": chunk_name, "model": model_name, "qid": q["qid"],
            "q_type": q["q_type"], "question": q["question"], "rank": rank,
            "top1_score": round(first_score, 3) if first_doc else None,
            "top1_title": first_doc.metadata.get("title") if first_doc else None,
        })

    total = len(questions)
    summary = {
        "chunk_set": chunk_name,
        "model": model_name,
        "dim": len(embedding_model.embed_query("테스트")),
        "Top-1": round(hits[1] / total, 3),
        "Top-3": round(hits[3] / total, 3),
        "Top-5": round(hits[5] / total, 3),
        "MRR": round(rr_sum / total, 3),
        "적재(초)": round(build_sec, 1),
        "검색(초/질문)": round((time.time() - t1) / total, 2),
    }
    return summary, details

## 7. 전체 실행 + 결과표 (청크 버전별, MRR 높은 순)

In [ ]:
import pandas as pd

summaries, all_details = [], []
for chunk_name in CHUNK_SETS:
    for model_name, embedding_model in models.items():
        print(f"평가 중: [{chunk_name}] {model_name}")
        summary, details = evaluate(model_name, embedding_model, chunk_name)
        summaries.append(summary)
        all_details.extend(details)

result_df = pd.DataFrame(summaries).sort_values(["chunk_set", "MRR"], ascending=[True, False])
result_df

## 8. 질문 유형별 정답률
의미검색 · 지역필터 · PDF근거별로 K등 안에 정답이 나온 비율 → 모델의 약점 확인

In [ ]:
detail_df = pd.DataFrame(all_details)
detail_df["hit"] = detail_df["rank"].notna()      # K등 안에 정답이 있었는지

detail_df.pivot_table(index=["chunk_set", "model"], columns="q_type", values="hit", aggfunc="mean").round(2)

## 9. 틀린 질문 보기 + 결과 저장
- 틀린 질문을 보면 모델의 약점을 알 수 있다
- CSV는 테스트 보고서에 그대로 사용 (`utf-8-sig` = 엑셀에서 한글 안 깨짐)

In [ ]:
import os

display(detail_df[~detail_df["hit"]][["chunk_set", "model", "qid", "question", "top1_title"]])

os.makedirs("./results", exist_ok=True)
result_df.to_csv("./results/ab_embedding_summary.csv", index=False, encoding="utf-8-sig")
detail_df.to_csv("./results/ab_embedding_detail.csv", index=False, encoding="utf-8-sig")
print("저장 완료")

## 10. 결과 정리 (테스트 후 작성)
| 항목 | 내용 |
|---|---|
| 테스트한 청크 버전 | |
| 선택 모델 | |
| 벡터 길이 | |
| 선택 이유 | Top-1 / Top-3 / Top-5 / MRR / 속도 / 비용 |
| 약점 | (틀린 질문 유형) |

→ 다음: 고른 모델로 청크 크기 A/B 테스트 (`CHUNK_SETS`에 버전 추가) → 청크 확정 → `01_embedding_test.ipynb`의 `EMBED_MODEL` 변경 후 최종 임베딩 · 적재. 질문 임베딩과 Neo4j도 같은 모델을 쓴다.